In [ ]:
pip install pystac-client==0.6.0 ipywidgets==8.1.8 pystac-client==0.6.0 pandas==3.0.0 shapely==2.1.2 plotly==6.6.0 geopandas==1.1.3 gdal==3.12.2 rasterio==1.5.0

In [ ]:
import getpass
collgs_token = getpass.getpass("Enter your access token from https://keycloak.grid.cesnet.cz/token-portal/index.php")

In [ ]:
from pystac_client import Client
from STAC_functions import display_collection
stac_url = 'https://stac.cesnet.cz/'
client = Client.open(stac_url)
print(client.title)

collection_dropdown = display_collection(stac_url, 'CESNET')

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from datetime import date
from datetime import datetime

# Widgets
start_picker = widgets.DatePicker(description='Start Date', value=date.today())
end_picker = widgets.DatePicker(description='End Date', value=date.today())
limit_slider = widgets.IntSlider(value=10, min=1, max=100, step=1, description='Item limit:')

# Display UI
ui = widgets.VBox([start_picker, end_picker, limit_slider])
display(ui)

In [ ]:
bbox_prague = [14.22, 49.94, 14.71, 50.18]
bbox_cz = [12.09, 48.55, 18.86, 51.06]

start_date = start_picker.value.strftime("%Y-%m-%d")
end_date = end_picker.value.strftime("%Y-%m-%d")
#max_cloud = cloud_slider.value
limit = limit_slider.value
#print(f"Date: {start_date} to {end_date}, cloud cover < {max_cloud}%, limit = {limit}")

collection = collection_dropdown.value

datetime_range = f"{start_picker.value.strftime('%Y-%m-%d')}/{end_picker.value.strftime('%Y-%m-%d')}"
search = client.search(
    bbox = bbox_cz,
    max_items=limit,
    collections= collection,
    datetime = f"{start_date}/{end_date}"
)

items = search.item_collection()

print(f"We found {len(items)} matching items.")

In [ ]:
import geopandas as gpd
from STAC_functions import plot_mapbox
geo_df = gpd.GeoDataFrame.from_features(search.item_collection())
plot_mapbox(geo_df)

In [ ]:
from STAC_functions import show_geo_map
dropdown = show_geo_map(geo_df)

Downloading images

In [ ]:
import requests

product_url = next(
    (asset.href
     for name, asset in items[dropdown.value].assets.items()
     if "_TCI_10m" in name),
    None
)
response = requests.get(product_url,
                        headers={"Authorization": f"Bearer {collgs_token}"})
if response.status_code == 200:
    print("URL is valid:", product_url)
    print("✅ File is accessible.")
else:
    print(f"❌ Failed to access URL. Status code: {response.status_code}")

In [ ]:
import os

downloads_path = "./downloads/"

if os.path.isdir(downloads_path):
    print("✅ downloads folder exists.")
else:
    os.makedirs(downloads_path)
    print("📁 downloads folder created.")

In [ ]:
from tqdm.notebook import tqdm
download_url = product_url
response = requests.get(download_url,
                        headers={"Authorization": f"Bearer {collgs_token}"}, stream=True)
response.raise_for_status()

# Start the download with a progress bar
total_size = int(response.headers.get('content-length', 0))
block_size = 1024

image_file = os.getcwd() + '/downloads/TCIimage_file.jp2' 

with open(image_file, 'wb') as file, tqdm(
    desc=f"Downloading",
    total=total_size,
    unit='iB',
    unit_scale=True,
    unit_divisor=1024,
) as bar:
    for data in response.iter_content(block_size):
        file.write(data)
        bar.update(len(data))

print(f"\n✅ File downloaded to: {image_file}")

In [ ]:
from PIL import Image
img = Image.open("downloads/TCIimage_file.jp2")
img.load()
display(img)